### Data Cleaning - Uber/Rapido Driver Dataset

This notebook cleans the `uber_rapido_driver_dirty.csv` file step by step.
The file has 50,000 rows and 33 columns about driver applications and
performance, and it has a lot of common data problems: missing values,
duplicate rows, inconsistent spelling/capitalization, wrong data types, a
messy phone/email/date columns, and some impossible values (like an age of
999 or a rating of -2).

Steps Performed:
1. Data quality report (before)
2. Duplicate row removal
3. Drop rows missing an ID (Primary key handling)
4. Standardization (text, categories, booleans, phone, email, dates)
5. Handle boolean values 
6. Missing-data imputation (documented per column)
7. Range validation (turning impossible/sentinel values into true nulls)
8. Filling placeholder values
9. Outlier Detection (IQR Method)
10. DataType correction
11. Final verification for duplicates 
12. Data quality report (after cleaning)
13. Before vs After data summary
14. Export cleaned dataset

Every non-trivial decision is justified in the markdown cell above the code that implements it.

In [43]:
#Install libraries and import dataset
import pandas as pd
import numpy as np

df = pd.read_csv('../data/uber_rapido_driver_dirty.csv')
raw = df.copy()   # keeping an untouched copy so I can compare before/after later

print(df.shape)
df.head(10)

(50000, 33)


,driver_id,driver_name,age,city,phone,email,platform,vehicle_type,vehicle_condition,experience_level,...,driving_test_score_10,route_knowledge_score_10,customer_handling_score_10,overall_interview_score_10,document_verification,background_check_status,interview_result,rejection_reason,onboarding_date,is_currently_active
0,DRV016649,Naresh Chauhan,NaN,Chennai,9255500096,naresh.chauhan@hotmail.com,uber,auto,good,0-1 yr,...,4.3,5.8,7.7,5.3,Fail,NaN,On Hold,NaN,-,false
1,DRV032785,Amit Gupta,45,Ahmedabad,9940345570,amit.gupta@rediffmail.com,Uber,Mini,Excellent,Fresher,...,9.3,NaN,9.0,8.1,NaN,Clear,On Hold,NaN,-,No
2,DRV011323,Yogesh Nair,53,Lucknow,9170487337,NaN,uber,SUV,avg,0-1 yr,...,6.3,9.0,5.5,6.9,NaN,clear,Selected,NaN,19 April 2021,false
3,DRV022168,Manoj Patel,46,Hyderabad,9246425981,manoj.patel@hotmail.com,RAPIDO,Mini,NaN,1-3 yrs,...,8.7,9.6,5.3,7.8,unknown,Clear,Selected,NaN,2021/05/27,0
4,DRV040782,Vikram Chaudhary,25,KOLKATA,9282240,vikram.chaudhary@gmail.com,Uber,suv,GOOD,Fresher,...,12.7,6.5,7.9,6.0,FAIL,clear,Rejected,Low rating history,-,NO
5,DRV003740,Sanjay Chaudhary,-32,Pune,+91-9120959146,sanjay.chaudhary@yahoo.com,Uber & Rapido,BIKE,excellent,Fresher,...,8.8,8.6,10.5,-1.4,FAIL,NaN,Selected,NaN,April 2023,N
6,DRV037372,Rakesh Chaudhary,21,Lucknow,+91-9109648055,rakesh.chaudhary@gmail.com,Uber & Rapido,BIKE,Average,0-1 yr,...,8.1,8.7,9.7,8.2,NaN,pending,Selected,NaN,08-08-2023,true
7,DRV042083,Naveen Mishra,999,Delhi,9988246,naveen.mishra@hotmail.com,Uber,suv,Excellent,0-1 yr,...,8.8,6.4,NaN,6.0,NaN,Clear,Selected,NaN,2022/02/06,0
8,DRV030189,Hitesh Singh,43,Kolkata,9994609866,NaN,UBER,Mini,Excellent,5+ yrs,...,-1.1,7.9,10.7,8.3,FAIL,NaN,Selected,NaN,2022-06-08T00:00:00Z,1
9,DRV029154,Harish Chauhan,39,Chennai,9712392703,harish.chauhan@gmail.com,UBER,Bike,GOOD,0-1 yr,...,11.6,7.7,5.9,6.7,FAIL,Flagged,On Hold,NaN,-,N


### 1. Data Quality Report (before cleaning)

Checking how many missing values each column has, how many duplicate rows
there are, and what data types pandas guessed when it loaded the file.

In [44]:
#Null values in the dataset
df.isnull().sum()

driver_id                       995
driver_name                       0
age                            2457
city                              0
phone                          3911
email                          4605
platform                       2239
vehicle_type                   2225
vehicle_condition             10532
experience_level               2224
preferred_shift               11227
reason_for_joining            11363
languages_known                2153
overall_driver_rating          3967
total_trips_completed          4070
trip_completion_rate_%         4435
cancellation_rate_%            4516
avg_daily_earnings_INR         2511
total_complaints               4035
total_compliments              3976
peak_hour_availability            0
interview_date                 4933
communication_score_10         4584
driving_test_score_10          4441
route_knowledge_score_10       4552
customer_handling_score_10     4500
overall_interview_score_10     4455
document_verification       

In [45]:
#Duplicate rows and missing values
print("Duplicate rows:", df.duplicated().sum())
print("Total missing values:", df.isnull().sum().sum())
df.dtypes

Duplicate rows: 951
Total missing values: 202535


driver_id                         str
driver_name                       str
age                               str
city                              str
phone                             str
email                             str
platform                          str
vehicle_type                      str
vehicle_condition                 str
experience_level                  str
preferred_shift                   str
reason_for_joining                str
languages_known                   str
overall_driver_rating             str
total_trips_completed         float64
trip_completion_rate_%        float64
cancellation_rate_%           float64
avg_daily_earnings_INR            str
total_complaints              float64
total_compliments             float64
peak_hour_availability            str
interview_date                    str
communication_score_10        float64
driving_test_score_10         float64
route_knowledge_score_10      float64
customer_handling_score_10    float64
overall_inte

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   driver_id                   49005 non-null  str    
 1   driver_name                 50000 non-null  str    
 2   age                         47543 non-null  str    
 3   city                        50000 non-null  str    
 4   phone                       46089 non-null  str    
 5   email                       45395 non-null  str    
 6   platform                    47761 non-null  str    
 7   vehicle_type                47775 non-null  str    
 8   vehicle_condition           39468 non-null  str    
 9   experience_level            47776 non-null  str    
 10  preferred_shift             38773 non-null  str    
 11  reason_for_joining          38637 non-null  str    
 12  languages_known             47847 non-null  str    
 13  overall_driver_rating       46033 non-null

In [47]:
# a quick look at min/max for the numeric columns to spot anything weird
df.describe()

,total_trips_completed,trip_completion_rate_%,cancellation_rate_%,total_complaints,total_compliments,communication_score_10,driving_test_score_10,route_knowledge_score_10,customer_handling_score_10,overall_interview_score_10
count,45930.000000,45565.00000,45484.000000,45965.000000,46024.000000,45416.000000,45559.000000,45448.000000,45500.000000,45545.000000
mean,6802.510799,77.89254,22.416584,8809.843120,8942.686272,6.508777,6.946878,6.508619,6.517708,6.613859
std,17638.171153,20.80393,23.708155,28303.698509,28456.710282,2.789817,2.630277,2.794014,2.804085,2.239486
min,-7996.000000,-2.00000,-2.000000,0.000000,0.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000
25%,1704.000000,69.40000,8.100000,14.000000,27.000000,4.600000,5.400000,4.600000,4.600000,5.900000
50%,3909.000000,80.50000,18.100000,27.000000,55.000000,6.600000,7.100000,6.600000,6.600000,6.700000
75%,6086.750000,91.60000,27.900000,41.000000,82.000000,8.500000,8.800000,8.600000,8.600000,7.400000
max,99999.000000,103.00000,103.000000,99999.000000,99999.000000,13.000000,13.000000,13.000000,13.000000,13.000000


Looking at `.describe()`, a few things stand out right away:
- `age` has a max of 999 - that's clearly a placeholder, not a real age
- `overall_driver_rating` goes below 0 and above 5, but it's supposed to be a 0-5 rating
- some score columns go above 10, but they're supposed to be out of 10
- `trip_completion_rate_%` and `cancellation_rate_%` go above 100%

I'll fix these later in the numeric cleaning step.

### 2. Remove Duplicate Rows

In [48]:
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (49049, 33)


### 3. Drop Rows Missing a `driver_id`

`driver_id` is the ID column (primary key), so a row without one isn't really usable - we
can't make up a new ID for someone, so I just dropped these rows.

In [49]:
df = df.dropna(subset=['driver_id'])
print("Shape after dropping rows with no driver_id:", df.shape)

Shape after dropping rows with no driver_id: (48078, 33)


### 4. Standardization - Clean Up Text Columns

A lot of the text columns have the same value written in different ways,
like `"Uber"`, `"UBER"`, and `"uber"`, or placeholder junk like `"na"`,
`"unknown"`, `"-"`. I wrote one small function to clean this up: strip extra
spaces, and turn placeholder words into real missing values (NaN).

One exception: for `reason_for_joining` and `rejection_reason`, the word
`"none"` is actually a real answer (meaning "no particular reason"), not a
placeholder, so I kept it instead of treating it as missing.

In [50]:
# Formatting the text columns to make them consistent and easier to work with
junk_words = ['na', 'n/a', 'unknown', '-', '', 'nan', 'null', 'none']

def clean_text(col, keep_none=False):
    col = col.astype(str).str.strip()
    words = [w for w in junk_words if not (keep_none and w == 'none')]
    col = col.where(~col.str.lower().isin(words), np.nan)
    return col

df['driver_name'] = df['driver_name'].astype(str).str.strip().str.title()
df['city'] = clean_text(df['city']).str.title()
df['platform'] = clean_text(df['platform']).str.title()
df['vehicle_type'] = clean_text(df['vehicle_type']).str.title().replace({'Suv': 'SUV'})
df['vehicle_condition'] = clean_text(df['vehicle_condition']).str.title().replace({'Avg': 'Average'})
df['experience_level'] = clean_text(df['experience_level'])
df['preferred_shift'] = clean_text(df['preferred_shift']).str.title()
df['document_verification'] = clean_text(df['document_verification']).str.title()
df['background_check_status'] = clean_text(df['background_check_status']).str.title()
df['interview_result'] = clean_text(df['interview_result']).str.title()
df['languages_known'] = clean_text(df['languages_known'])

# "none" is a real answer for these two columns, so we keep it
df['reason_for_joining'] = clean_text(df['reason_for_joining'], keep_none=True)
df['rejection_reason'] = clean_text(df['rejection_reason'], keep_none=True)

df['platform'].unique()

<StringArray>
['Uber', 'Rapido', 'Uber & Rapido', nan, 'Ola']
Length: 5, dtype: str

In [51]:
df['vehicle_type'].unique()

<StringArray>
['Auto', 'Mini', 'SUV', 'Bike', 'Sedan', nan]
Length: 6, dtype: str

### 5. Fix Yes/No Columns

`peak_hour_availability` and `is_currently_active` have a mix of
`Yes/No`, `Y/N`, `True/False`, and `1/0`. I mapped all of these to a proper
`True`/`False` boolean.

In [52]:
# Handling the boolean columns by mapping yes/no values to True/False
yes_no_map = {'yes': True, 'y': True, 'true': True, '1': True,
              'no': False, 'n': False, 'false': False, '0': False}

df['peak_hour_availability'] = df['peak_hour_availability'].astype(str).str.lower().map(yes_no_map)
df['is_currently_active'] = df['is_currently_active'].astype(str).str.lower().map(yes_no_map)

df[['peak_hour_availability', 'is_currently_active']].head(10)

,peak_hour_availability,is_currently_active
0,False,False
1,True,False
2,False,False
3,False,False
4,True,False
5,False,False
6,True,True
7,True,False
8,True,True
9,True,False


### 6. Missing-data imputation (documented per column) 
**Handling Phone Numbers**

I kept only the digits from each phone number. A real Indian mobile number
has 10 digits, so I kept those, and also handled numbers that had the `91`
country code in front of them (12 digits total). Anything else (like the
`XXXXXXXXXX` placeholder, or a 7-digit fragment) becomes missing - I can't
guess someone's real phone number, so it's better to leave it blank.

In [53]:
def clean_phone(x):
    if pd.isna(x):
        return np.nan
    digits = ''.join(ch for ch in x if ch.isdigit())
    if len(digits) == 12 and digits.startswith('91'):
        digits = digits[2:]
    if len(digits) == 10:
        return digits
    return np.nan

df['phone'] = df['phone'].apply(clean_phone)
print("Valid phone numbers:", df['phone'].notna().sum(), "out of", len(df))

Valid phone numbers: 39538 out of 48078


**Handling E-mails**

For emails I did a simple check - does it contain an `@` and a `.`? If not,
it's not a usable email address, so I set it to missing rather than trying
to guess what the real address was supposed to be.

In [54]:
df['email'] = df['email'].astype(str).str.strip().str.lower()
has_at = df['email'].str.contains('@', na=False)
has_dot = df['email'].str.contains('.', na=False, regex=False)
df.loc[~(has_at & has_dot), 'email'] = np.nan

print("Valid emails:", df['email'].notna().sum(), "out of", len(df))

Valid emails: 41702 out of 48078


**Fixing the Date Columns**

`interview_date` and `onboarding_date` have several different date formats
mixed together in the same column (e.g. `"10 July 2023"`, `"2023/03/13"`,
`"08.11.2021"`, `"June 2021"`, and an ISO format like
`"2023-06-25T00:00:00Z"`). There's also a `"-"` used for missing dates.

Instead of writing complicated code to detect each format, I just tried
converting the column using each known format one at a time with
`pd.to_datetime()`. Any date that doesn't match a given format just comes
back empty (`NaT`), so I filled those gaps in with the result from trying
the next format.

In [55]:
def parse_dates(col):
    d1 = pd.to_datetime(col, format='%Y-%m-%dT%H:%M:%SZ', errors='coerce')
    d2 = pd.to_datetime(col, format='%d %B %Y', errors='coerce')
    d3 = pd.to_datetime(col, format='%B %Y', errors='coerce')       # e.g. "June 2021" -> defaults to the 1st
    d4 = pd.to_datetime(col, format='%Y/%m/%d', errors='coerce')
    d5 = pd.to_datetime(col, format='%d.%m.%Y', errors='coerce')
    d6 = pd.to_datetime(col, format='%d-%m-%Y', errors='coerce')
    d7 = pd.to_datetime(col, format='%m/%d/%y', errors='coerce')
    return d1.fillna(d2).fillna(d3).fillna(d4).fillna(d5).fillna(d6).fillna(d7)

df['interview_date'] = parse_dates(df['interview_date'])
df['onboarding_date'] = parse_dates(df['onboarding_date'])

print("interview_date - valid dates:", df['interview_date'].notna().sum())
print("onboarding_date - valid dates:", df['onboarding_date'].notna().sum())

interview_date - valid dates: 43346
onboarding_date - valid dates: 18231


I'm not filling in the missing dates. We can't know what date someone was
actually interviewed or onboarded on, so guessing a date would just make the
data less trustworthy, not more complete. Same reasoning for phone and email
above - some things shouldn't be guessed.

### 7. Range validation (turning impossible/sentinel values into true nulls)

Now I'll fix the number columns. Each of these has a valid range it's
supposed to fall in (e.g. a rating out of 5, a percentage out of 100), so
anything outside that range gets turned into a missing value - it must be a
data entry mistake, not a real value.

In [56]:
# age should be between 18 and 65 for a driver
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df.loc[(df['age'] < 18) | (df['age'] > 65), 'age'] = np.nan

# earnings: strip "Rs.", "/-", and commas, then convert to numbers
df['avg_daily_earnings_INR'] = (df['avg_daily_earnings_INR'].astype(str)
                                 .str.replace('Rs.', '', regex=False)
                                 .str.replace('/-', '', regex=False)
                                 .str.replace(',', '', regex=False))
df['avg_daily_earnings_INR'] = pd.to_numeric(df['avg_daily_earnings_INR'], errors='coerce')
# earnings can't be negative, and 9999999 is clearly a placeholder value
df.loc[(df['avg_daily_earnings_INR'] < 0) | (df['avg_daily_earnings_INR'] == 9999999), 'avg_daily_earnings_INR'] = np.nan

# rates should be between 0 and 100%
for col in ['trip_completion_rate_%', 'cancellation_rate_%']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[(df[col] < 0) | (df[col] > 100), col] = np.nan

# interview scores should be between 0 and 10
for col in ['communication_score_10', 'driving_test_score_10', 'route_knowledge_score_10',
            'customer_handling_score_10', 'overall_interview_score_10']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[(df[col] < 0) | (df[col] > 10), col] = np.nan

# driver rating should be between 0 and 5
df['overall_driver_rating'] = pd.to_numeric(df['overall_driver_rating'], errors='coerce')
df.loc[(df['overall_driver_rating'] < 0) | (df['overall_driver_rating'] > 5), 'overall_driver_rating'] = np.nan

# these counts can't be negative, and 99999 is a placeholder value
for col in ['total_trips_completed', 'total_complaints', 'total_compliments']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[(df[col] < 0) | (df[col] == 99999), col] = np.nan

df[['age', 'overall_driver_rating', 'avg_daily_earnings_INR']].describe()

,age,overall_driver_rating,avg_daily_earnings_INR
count,38922.000000,38441.00000,41829.000000
mean,37.536869,3.74836,1443.372469
std,10.359971,0.72347,604.958819
min,20.000000,2.50000,400.000000
25%,29.000000,3.10000,922.000000
50%,38.000000,3.70000,1437.000000
75%,46.000000,4.40000,1964.000000
max,55.000000,5.00000,2500.000000


### 8. Fill In Missing Values

Now that the columns are actually clean (real missing values instead of junk
text), here's how I filled each type of column in:

- **Number columns** (age, earnings, scores, etc.) -> filled with the
  **median**, since it's not affected much by a few extreme values
- **Category columns** (platform, vehicle type, etc.) -> filled with the
  **most common value (mode)**, since that's a reasonable guess for a
  missing category
- **`reason_for_joining` / `rejection_reason`** -> filled with a clear label
  ("Not Specified" / "Not Applicable") instead of the mode, since a missing
  value here usually just means the question didn't apply to that driver
- **Phone, email, interview date, onboarding date** -> left blank on purpose.
  These are things we genuinely can't guess for someone (see Steps 6-8).

In [57]:
# Range checks and cleaning for numeric columns, filling missing values with median for numeric and mode for categorical columns
numeric_cols = ['age', 'avg_daily_earnings_INR', 'total_trips_completed',
                'trip_completion_rate_%', 'cancellation_rate_%', 'total_complaints',
                'total_compliments', 'communication_score_10', 'driving_test_score_10',
                'route_knowledge_score_10', 'customer_handling_score_10',
                'overall_interview_score_10', 'overall_driver_rating']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

category_cols = ['platform', 'vehicle_type', 'vehicle_condition', 'experience_level',
                  'preferred_shift', 'document_verification', 'background_check_status',
                  'interview_result', 'languages_known']
for col in category_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df['reason_for_joining'] = df['reason_for_joining'].fillna('Not Specified')
df['rejection_reason'] = df['rejection_reason'].fillna('Not Applicable')

df.isnull().sum()

driver_id                         0
driver_name                       0
age                               0
city                              0
phone                          8540
email                          6376
platform                          0
vehicle_type                      0
vehicle_condition                 0
experience_level                  0
preferred_shift                   0
reason_for_joining                0
languages_known                   0
overall_driver_rating             0
total_trips_completed             0
trip_completion_rate_%            0
cancellation_rate_%               0
avg_daily_earnings_INR            0
total_complaints                  0
total_compliments                 0
peak_hour_availability            0
interview_date                 4732
communication_score_10            0
driving_test_score_10             0
route_knowledge_score_10          0
customer_handling_score_10        0
overall_interview_score_10        0
document_verification       

### 9. Outlier Detection (IQR Method)

Using the standard IQR rule: anything below `Q1 - 1.5*IQR` or above
`Q3 + 1.5*IQR` counts as an outlier. Instead of deleting these rows (which
would lose data), I capped them at the lower/upper bound using `.clip()`.

In [58]:
outlier_summary = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    # Cap values
    df[col] = df[col].clip(lower, upper)

    outlier_summary.append({
        "Column": col,
        "Q1 (25%)": round(q1, 2),
        "Q3 (75%)": round(q3, 2),
        "IQR": round(iqr, 2),
        "Lower Bound": round(lower, 2),
        "Upper Bound": round(upper, 2),
        "Outliers Found": n_outliers,
        "Action": "Capped to IQR range"
    })

# Create detailed table
outlier_table = pd.DataFrame(outlier_summary)

# Display table
display(outlier_table)

,Column,Q1 (25%),Q3 (75%),IQR,Lower Bound,Upper Bound,Outliers Found,Action
0,age,31.0,44.0,13.0,11.50,63.50,0,Capped to IQR range
1,avg_daily_earnings_INR,998.0,1887.0,889.0,-335.50,3220.50,0,Capped to IQR range
2,total_trips_completed,2361.0,5589.0,3228.0,-2481.00,10431.00,0,Capped to IQR range
3,trip_completion_rate_%,72.6,87.5,14.9,50.25,109.85,0,Capped to IQR range
4,cancellation_rate_%,10.9,24.2,13.3,-9.05,44.15,0,Capped to IQR range
5,total_complaints,15.0,35.0,20.0,-15.00,65.00,0,Capped to IQR range
6,total_compliments,29.0,70.0,41.0,-32.50,131.50,0,Capped to IQR range
7,communication_score_10,5.2,7.8,2.6,1.30,11.70,0,Capped to IQR range
8,driving_test_score_10,5.8,8.1,2.3,2.35,11.55,0,Capped to IQR range
9,route_knowledge_score_10,5.1,7.8,2.7,1.05,11.85,0,Capped to IQR range


In [59]:
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_outliers} outliers found, capped to range ({lower:.1f}, {upper:.1f})")
    df[col] = df[col].clip(lower, upper)

age: 0 outliers found, capped to range (11.5, 63.5)
avg_daily_earnings_INR: 0 outliers found, capped to range (-335.5, 3220.5)
total_trips_completed: 0 outliers found, capped to range (-2481.0, 10431.0)
trip_completion_rate_%: 0 outliers found, capped to range (50.2, 109.9)
cancellation_rate_%: 0 outliers found, capped to range (-9.0, 44.1)
total_complaints: 0 outliers found, capped to range (-15.0, 65.0)
total_compliments: 0 outliers found, capped to range (-32.5, 131.5)
communication_score_10: 0 outliers found, capped to range (1.3, 11.7)
driving_test_score_10: 0 outliers found, capped to range (2.4, 11.5)
route_knowledge_score_10: 0 outliers found, capped to range (1.0, 11.9)
customer_handling_score_10: 0 outliers found, capped to range (1.1, 12.0)
overall_interview_score_10: 0 outliers found, capped to range (4.6, 8.6)
overall_driver_rating: 0 outliers found, capped to range (1.9, 5.6)


Most columns had zero or very few outliers once the impossible values from
Step 7 were already removed. `overall_interview_score_10` had more outliers
than the others - this is mostly because filling in missing scores with the
median (Step 8) pulled a lot of values toward the center, which naturally
makes anything further away look more like an outlier by comparison. I
capped these too rather than removing rows, so no data is lost.

### 10. Data Types Correction

Setting each column to the type it should actually be: whole numbers as
`int`, decimals as `float`, dates as `datetime`, and the two yes/no columns
as `bool`.

In [60]:
df['driver_id'] = df['driver_id'].astype(str)
df['driver_name'] = df['driver_name'].astype(str)
df['age'] = df['age'].astype(int)
df['total_trips_completed'] = df['total_trips_completed'].astype(int)
df['total_complaints'] = df['total_complaints'].astype(int)
df['total_compliments'] = df['total_compliments'].astype(int)
df['avg_daily_earnings_INR'] = df['avg_daily_earnings_INR'].round(2)
df['trip_completion_rate_%'] = df['trip_completion_rate_%'].round(1)
df['cancellation_rate_%'] = df['cancellation_rate_%'].round(1)
df['overall_driver_rating'] = df['overall_driver_rating'].round(1)
df['peak_hour_availability'] = df['peak_hour_availability'].astype(bool)
df['is_currently_active'] = df['is_currently_active'].astype(bool)
# interview_date and onboarding_date are already proper dates from Step 8

df.dtypes

driver_id                                str
driver_name                              str
age                                    int64
city                                     str
phone                                    str
email                                    str
platform                                 str
vehicle_type                             str
vehicle_condition                        str
experience_level                         str
preferred_shift                          str
reason_for_joining                       str
languages_known                          str
overall_driver_rating                float64
total_trips_completed                  int64
trip_completion_rate_%               float64
cancellation_rate_%                  float64
avg_daily_earnings_INR               float64
total_complaints                       int64
total_compliments                      int64
peak_hour_availability                  bool
interview_date                datetime64[us]
communicat

### 11. Final Verification for Duplicate Values

After cleaning up all the spelling/capitalization/date format issues, some
rows that used to look different (e.g. one said `"UBER"` and the other said
`"uber"`) might now be exact duplicates of each other. So I ran
`drop_duplicates()` one more time just to be safe.

In [61]:
rows_before = len(df)
df = df.drop_duplicates()
print("Extra duplicates removed after cleaning:", rows_before - len(df))
print("Final row count:", len(df))

Extra duplicates removed after cleaning: 476
Final row count: 47602


### 12. Data Quality Report (after cleaning)

In [62]:
df.isnull().sum()

driver_id                         0
driver_name                       0
age                               0
city                              0
phone                          8468
email                          6314
platform                          0
vehicle_type                      0
vehicle_condition                 0
experience_level                  0
preferred_shift                   0
reason_for_joining                0
languages_known                   0
overall_driver_rating             0
total_trips_completed             0
trip_completion_rate_%            0
cancellation_rate_%               0
avg_daily_earnings_INR            0
total_complaints                  0
total_compliments                 0
peak_hour_availability            0
interview_date                 4686
communication_score_10            0
driving_test_score_10             0
route_knowledge_score_10          0
customer_handling_score_10        0
overall_interview_score_10        0
document_verification       

In [63]:
print("Duplicate rows:", df.duplicated().sum())
print("Total missing values:", df.isnull().sum().sum())
df.dtypes

Duplicate rows: 0
Total missing values: 49019


driver_id                                str
driver_name                              str
age                                    int64
city                                     str
phone                                    str
email                                    str
platform                                 str
vehicle_type                             str
vehicle_condition                        str
experience_level                         str
preferred_shift                          str
reason_for_joining                       str
languages_known                          str
overall_driver_rating                float64
total_trips_completed                  int64
trip_completion_rate_%               float64
cancellation_rate_%                  float64
avg_daily_earnings_INR               float64
total_complaints                       int64
total_compliments                      int64
peak_hour_availability                  bool
interview_date                datetime64[us]
communicat

### 13. Before vs. After Summary

In [64]:
summary = pd.DataFrame({
    'Metric': ['Rows', 'Total missing values', 'Duplicate rows'],
    'Before': [len(raw), raw.isnull().sum().sum(), raw.duplicated().sum()],
    'After': [len(df), df.isnull().sum().sum(), df.duplicated().sum()],
})
summary

,Metric,Before,After
0,Rows,50000,47602
1,Total missing values,202535,49019
2,Duplicate rows,951,0


The row count dropped a bit from the original 50,000 - that's from
removing duplicate rows (both the obvious ones and the ones that only became
obvious after cleaning) and rows that were missing a driver ID. The missing
values remaining in the "After" column are all on purpose (phone, email, and
the two date columns) - I explained why in Steps 6-8 instead of just making
up values to fill every cell.

### 14. Save the Cleaned Dataset

In [65]:
df.to_csv('uber_rapido_driver_clean.csv', index=False)
print("Saved! Final shape:", df.shape)
df.head(10)

Saved! Final shape: (47602, 33)


,driver_id,driver_name,age,city,phone,email,platform,vehicle_type,vehicle_condition,experience_level,...,driving_test_score_10,route_knowledge_score_10,customer_handling_score_10,overall_interview_score_10,document_verification,background_check_status,interview_result,rejection_reason,onboarding_date,is_currently_active
0,DRV016649,Naresh Chauhan,38,Chennai,9255500096,naresh.chauhan@hotmail.com,Uber,Auto,Good,0-1 yr,...,4.3,5.8,7.7,5.3,Fail,Clear,On Hold,Not Applicable,NaT,False
1,DRV032785,Amit Gupta,45,Ahmedabad,9940345570,amit.gupta@rediffmail.com,Uber,Mini,Excellent,Fresher,...,9.3,6.5,9.0,8.1,Fail,Clear,On Hold,Not Applicable,NaT,False
2,DRV011323,Yogesh Nair,53,Lucknow,9170487337,NaN,Uber,SUV,Average,0-1 yr,...,6.3,9.0,5.5,6.9,Fail,Clear,Selected,Not Applicable,2021-04-19,False
3,DRV022168,Manoj Patel,46,Hyderabad,9246425981,manoj.patel@hotmail.com,Rapido,Mini,Good,1-3 yrs,...,8.7,9.6,5.3,7.8,Fail,Clear,Selected,Not Applicable,2021-05-27,False
4,DRV040782,Vikram Chaudhary,25,Kolkata,NaN,vikram.chaudhary@gmail.com,Uber,SUV,Good,Fresher,...,7.0,6.5,7.9,6.0,Fail,Clear,Rejected,Low rating history,NaT,False
5,DRV003740,Sanjay Chaudhary,38,Pune,9120959146,sanjay.chaudhary@yahoo.com,Uber & Rapido,Bike,Excellent,Fresher,...,8.8,8.6,6.5,6.6,Fail,Clear,Selected,Not Applicable,2023-04-01,False
6,DRV037372,Rakesh Chaudhary,21,Lucknow,9109648055,rakesh.chaudhary@gmail.com,Uber & Rapido,Bike,Average,0-1 yr,...,8.1,8.7,9.7,8.2,Fail,Pending,Selected,Not Applicable,2023-08-08,True
7,DRV042083,Naveen Mishra,38,Delhi,NaN,naveen.mishra@hotmail.com,Uber,SUV,Excellent,0-1 yr,...,8.8,6.4,6.5,6.0,Fail,Clear,Selected,Not Applicable,2022-02-06,False
8,DRV030189,Hitesh Singh,43,Kolkata,9994609866,NaN,Uber,Mini,Excellent,5+ yrs,...,7.0,7.9,6.5,8.3,Fail,Clear,Selected,Not Applicable,2022-06-08,True
9,DRV029154,Harish Chauhan,39,Chennai,9712392703,harish.chauhan@gmail.com,Uber,Bike,Good,0-1 yr,...,7.0,7.7,5.9,6.7,Fail,Flagged,On Hold,Not Applicable,NaT,False


### Summary

- Removed duplicate rows (951 obvious ones, plus 476 more that only matched
  after cleaning up the text formatting)
- Dropped rows with no driver ID (can't work with an unidentified record)
- Fixed inconsistent spelling/capitalization across all the text columns
- Cleaned up phone numbers, emails, and 7 different date formats
- Turned impossible values (age of 999, rating of -2, etc.) into real
  missing values, then filled them in with the median (numbers) or the most
  common value (categories)
- Checked for outliers using the IQR method and capped the extreme values
- Fixed all the data types
- Saved the cleaned dataset as `uber_rapido_driver_clean.csv`